# RG3 → CN7 RH 가스 직접 전이
CN7을 학습·기준보정에서 제외하고 기존 RG3 모델을 그대로 적용한다.

In [1]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
prefix=''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0]
exec(prefix)
source_n=len(d)
raw=pd.read_csv(root/'data/original_split/moldset_labeled_cn7.csv',parse_dates=['TimeStamp'])
raw['product']='CN7';raw['Reason']=raw.Reason.fillna('정상')
raw=raw.loc[raw.PART_NAME.str.contains(r'RH\s*$',regex=True)&~raw.PART_FACT_PLAN_DATE.str.startswith('2020-10-27')]
raw=raw.loc[raw.PassOrFail.eq(0)|raw.Reason.eq('가스')]
keys=['EQUIP_CD','product','TimeStamp']
allshots=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
target=raw[keys+['Reason','PassOrFail']].merge(allshots.drop(columns=['Reason','PassOrFail']),on=keys,validate='one_to_one')
d=pd.concat([d,target],ignore_index=True)
y=d.PassOrFail.to_numpy(int);groups=d.comparison_block
train=np.arange(source_n);test=np.arange(source_n,len(d))
assert len(test)==603 and y[test].sum()==10 and set(d['product'].iloc[train])=={'RG3'} and set(d['product'].iloc[test])=={'CN7'}
# res_features uses source normal relationships; no target normal or gas is used.
outer=pair_predict(train,test)
inner=np.full((len(train),2),np.nan)
for block in groups.iloc[train].unique():
    mask=groups.iloc[train].eq(block).to_numpy()
    inner[mask]=pair_predict(train[~mask],train[mask])
out=[];preds=[]
for name,k in [('std4',0),('residual4',1),('max_union',None)]:
    score=outer[:,k] if k is not None else outer.max(axis=1)
    cal=inner[:,k] if k is not None else inner.max(axis=1)
    threshold=float(np.quantile(cal[y[train]==0],.99,method='higher'))
    for rule,pred in [('fixed05',score>=.5),('source_normal_fpr1',score>threshold)]:
        tn,fp,fn,tp=confusion_matrix(y[test],pred,labels=[0,1]).ravel()
        out.append(dict(model=name,rule=rule,source_product='RG3',train_shots=len(train),train_gas=int(y[train].sum()),target_product='CN7_RH',test_shots=len(test),test_gas=int(y[test].sum()),threshold=.5 if rule=='fixed05' else threshold,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),AP=average_precision_score(y[test],score)))
        for i,j in enumerate(test):preds.append(dict(model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[j],actual=int(y[j]),score=float(score[i]),prediction=int(pred[i])))
mt=pd.DataFrame(out)
mt.to_csv(tables/'rg3_to_cn7_zero_shot_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(preds).to_csv(tables/'rg3_to_cn7_zero_shot_predictions.csv',index=False,encoding='utf-8-sig')
lines=['# RG3만 학습 → CN7 RH 가스 직접 적용','',
'- 학습: RG3 정상566+가스17. 평가: CN7 RH 정상593+가스10, 10/27초기기간 제외. LH가스3개는대상아님.',
'- CN7라벨·센서분포를학습또는임계값조정에사용하지않았다. CN7값은현재평가입력과그원본과거이력파생값에만사용.',
'- 기존흔들림RF4개·관계잔차RF4개·최댓값결합을사용. 센서관계Ridge는RG3정상으로만학습해서CN7에그대로적용.',
'- 고정0.5와RG3학습정상의안쪽검증99%경계두기준만비교. CN7결과를보고최적임계값을고르지않음.',
'- RG3는CN7평가기간보다뒤에생산돼이실험은제품간전이진단이지시간순미래생산시험이아님.',
'- 반복탐색의동일CN7평가자료이므로독립최종테스트로주장하지않음.',
'','| model | rule | threshold | TP | FP | FN | TN | AP |','|---|---|---:|---:|---:|---:|---:|---:|']
for r in mt.itertuples():lines.append(f'| {r.model} | {r.rule} | {r.threshold:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.AP:.4f} |')
(root/'reports/rg3_to_cn7_zero_shot_transfer.md').write_text('\n'.join(lines),encoding='utf-8')
print(mt.to_string(index=False))


    model               rule source_product  train_shots  train_gas target_product  test_shots  test_gas  threshold  TP  FP  FN  TN       AP
     std4            fixed05            RG3          583         17         CN7_RH         603        10   0.500000   0   0  10 593 0.079287
     std4 source_normal_fpr1            RG3          583         17         CN7_RH         603        10   0.502745   0   0  10 593 0.079287
residual4            fixed05            RG3          583         17         CN7_RH         603        10   0.500000   0   0  10 593 0.056180
residual4 source_normal_fpr1            RG3          583         17         CN7_RH         603        10   0.374891   0   0  10 593 0.056180
max_union            fixed05            RG3          583         17         CN7_RH         603        10   0.500000   0   0  10 593 0.100562
max_union source_normal_fpr1            RG3          583         17         CN7_RH         603        10   0.506358   0   0  10 593 0.100562
